# 3D ViT v4 — within-region clusters

v3 detached the classifier from the 3D vector and trained the encoder with SupCon. Validation BCE fell. Validation SupCon rose back to chance. The two losses were not training the same weights, so a better classifier did not make two clusters.

This run keeps the same ViT, the same 9 PLS scores, the same 10 metros, and the same leave-one-metro-out split. The 3D output is the thing that has to cluster.

Each training batch is one metro, with an equal number of residential and non-residential tiles. On the raw encoder vectors in that batch:

$$
\mathcal{L} = \mathrm{BCE}\big(s \cdot (z \cdot \hat w),\ y\big) + \overline{\|z - c_y\|^2} + \mathrm{relu}(3 - \mathrm{sep}/\mathrm{within})^2
$$

$c_y$ is the mean of that class in the batch. sep is the distance between the two class means. within is the mean distance to the class mean. The gradient of BCE and of both cluster terms reaches $z$. $\hat w$ is a unit vector in the same 3D space, so residential stays on one side of the gap in every metro.

The number that stops training is the mean validation sep/within, averaged over the training metros in the 10% split. Higher is better. A fold always runs 35 epochs. After that it stops if the gap has not improved by 0.001 for 6 epochs. The cap is 60. F1 is logged and does not choose the checkpoint.


## What each log field means

One file per held-out metro: `outputs/week4_vit_100m_3d_v4/folds/<metro>/train.log`. The same rows are appended, during training, to `folds/<metro>/metrics.csv`.

| Field | Meaning | What to look for |
|---|---|---|
| bce | Classification loss. The head reads raw z | Should fall. Its gradient moves the encoder |
| pull | Mean squared distance to the class center | Should fall if each class tightens |
| short | relu(3 − sep/within)² | Reaches 0 when the gap is at least 3 |
| ratio, sep, within | Gap on the training batches | ratio is sep/within. The target is 3 |
| val_ratio | Mean sep/within on the validation metros | This is the stop metric. Higher is better |
| val_f1, prec, rec | Macro scores on 10% of the training metros | Logged only. Not the held-out city |
| best_ratio, bad | Best val gap and epochs since it improved | A stop is allowed only after epoch 35 |
| ckpt | Which files were written this epoch | `best` is the highest val gap. `last` is every epoch. `epoch_005` every fifth epoch |

Checkpoints for a fold live in `outputs/week4_vit_100m_3d_v4/folds/<metro>/` as `best.pt`, `last.pt`, and `epoch_005.pt` onward. Each file has the weights, optimizer, scheduler, epoch, best gap, early-stop count, seed, and RNG state. `last.pt` is resumable when the config matches. The driver log is `outputs/week4_vit_100m_3d_v4/train.log`.


## Code

The training driver is `scripts/train_week4_vit_100m_3d_v4.py`. The cell below is the cluster term. The ViT, the PLS projection, and the fold loop are in the driver.

```bash
PYTHONWARNINGS=ignore setsid nohup .venv/bin/python -u scripts/train_week4_vit_100m_3d_v4.py \
  > outputs/week4_vit_100m_3d_v4/train.log 2>&1 < /dev/null &
```


In [ ]:
"""v4 within-region cluster loss.

Matches scripts/train_week4_vit_100m_3d_v4.py.
Each batch is one metro. The gap is sep/within in the raw 3D output.
The run trains at least 35 epochs, then stops if that validation gap
has not improved for 6 epochs. The cap is 60. Higher is better.
"""
import torch
import torch.nn.functional as F

TARGET_RATIO = 3.0


def cluster_terms(z, y, target_ratio=TARGET_RATIO):
    labels = y.view(-1).long()
    centers, pulls, radii = [], [], []
    for k in (0, 1):
        pts = z[labels == k]
        if pts.shape[0] < 2:
            return None
        center = pts.mean(dim=0)
        centers.append(center)
        pulls.append((pts - center).pow(2).sum(dim=-1).mean())
        radii.append((pts - center).norm(dim=-1).mean())
    pull = torch.stack(pulls).mean()
    within = torch.stack(radii).mean()
    sep = (centers[1] - centers[0]).norm()
    ratio = sep / within.clamp_min(1e-6)
    short = F.relu(target_ratio - ratio).pow(2)
    return pull, short, ratio, sep, within


# logits, z = model(x, mask)     # z is not detached
# bce = BCEWithLogits(logits, y) # logits = scale * (z · unit w)
# pull, short, ratio, sep, within = cluster_terms(z, y)
# loss = bce + pull + short


## Results

LORO macro-F1 = **0.738** ± 0.125. Pooled OOF F1 = 0.760. v1 was 0.732. v3 was 0.735.

Mean held-out gap (sep/within) = **2.13**. A gap of 3 is the clear-separation line. Elapsed 829.9 s.

| Metro | F1 | sep | within | sep/within | best val gap | best epoch |
|---|---:|---:|---:|---:|---:|---:|
| Atlanta_SE | 0.898 | 0.39 | 0.11 | 3.51 | 2.69 | 21 |
| Phoenix_SW | 0.878 | 0.40 | 0.12 | 3.19 | 2.75 | 34 |
| Denver_W | 0.862 | 0.54 | 0.17 | 3.15 | 2.85 | 36 |
| Seattle_NW | 0.824 | 0.40 | 0.14 | 2.81 | 2.95 | 39 |
| LA_W | 0.793 | 0.37 | 0.15 | 2.50 | 3.00 | 27 |
| Houston_S | 0.709 | 0.28 | 0.14 | 1.96 | 2.94 | 35 |
| Boston_NE | 0.693 | 0.21 | 0.16 | 1.30 | 3.13 | 39 |
| NYC_NE | 0.586 | 0.09 | 0.13 | 0.63 | 3.33 | 25 |
| SF_W | 0.584 | 0.15 | 0.15 | 0.99 | 3.06 | 25 |
| Chicago_MW | 0.553 | 0.14 | 0.11 | 1.32 | 3.07 | 29 |

The 3D axes are the raw encoder outputs. Do not pool them across metros. Each fold has its own axes.

![Gap by metro](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_sep_ratio.png)

![2D projection](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_projection.png)

## Encoder output

### Atlanta SE

![3D Atlanta_SE](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_Atlanta_SE.png)

### Phoenix SW

![3D Phoenix_SW](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_Phoenix_SW.png)

### Denver W

![3D Denver_W](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_Denver_W.png)

### Seattle NW

![3D Seattle_NW](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_Seattle_NW.png)

### LA W

![3D LA_W](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_LA_W.png)

### Houston S

![3D Houston_S](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_Houston_S.png)

### Chicago MW

![3D Chicago_MW](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_Chicago_MW.png)

### Boston NE

![3D Boston_NE](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_Boston_NE.png)

### SF W

![3D SF_W](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_SF_W.png)

### NYC NE

![3D NYC_NE](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_NYC_NE.png)


